In [10]:
import re
from pathlib import Path

import nltk
from pypdf import PdfReader

import numpy as np

# 初回のみNLTKのデータセットをダウンロード
nltk.download("punkt")
nltk.download("punkt_tab")

# data/01_raw/pdfにあるPDFを読み込む
pdf_dir = Path("../../data/01_raw/pdf")
pdf_paths = sorted(pdf_dir.glob("*.pdf"))

print(f"読み込んだPDF数: {len(pdf_paths)}")

読み込んだPDF数: 2


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Owner\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Owner\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [11]:
def process_pdfs_to_sentences(pdf_list):
    all_sentences = []

    for i, pdf_path in enumerate(pdf_list):
        try:
            # PDFの各ページからテキストを抽出
            reader = PdfReader(str(pdf_path))
            raw_text = "\n".join(
                page.extract_text() or "" for page in reader.pages
            )
            if not raw_text.strip():
                raise ValueError(
                    "テキストを抽出できませんでした（画像PDFの可能性があります）。"
                )

            # NLTKによる文分割
            sentences = nltk.sent_tokenize(raw_text)

            # 各文を単語に分割して前処理
            for sent in sentences:
                clean_sent = re.sub(r"[^a-zA-Z\s]", "", sent).lower()
                word_list = nltk.word_tokenize(clean_sent)
                if word_list:
                    all_sentences.append(word_list)

            print(f"[{i + 1}/{len(pdf_list)}] 成功: {pdf_path.name}")

        except Exception as e:
            print(f"[{i + 1}/{len(pdf_list)}] エラー ({pdf_path.name}): {e}")

    return all_sentences


# 実行
dataset = process_pdfs_to_sentences(pdf_paths)

# 結果の確認
print(f"\n総文章数: {len(dataset)} 文")
print("1文目のサンプル:", dataset[0] if dataset else "なし")

[1/2] 成功: 10.pdf
[2/2] 成功: 11.pdf

総文章数: 1028 文
1文目のサンプル: ['scientia', 'horticulturae', 'available', 'online', 'may', 'elsevier', 'bv', 'all', 'rights', 'are', 'reserved', 'including', 'those', 'for', 'text', 'and', 'data', 'mining', 'ai', 'training', 'and', 'similar', 'technologies']


In [12]:
from gensim.models import Word2Vec

# そのままWord2Vecに投入
model = Word2Vec(
    sentences=dataset, vector_size=100, window=5, min_count=2, workers=4
)